# AI Influencer Studio — a free web app for AI-influencer videos

Turn a character, one photo and a script into a finished vertical talking video, in a point-and-click web app
that runs on Kaggle's free GPU, with open models and no API keys:
**character → idea → script → voice → moving video → lip-sync → subtitles and sound**.

| Step | Model | What it does |
|---|---|---|
| Ideas & script | Qwen3-4B-Instruct | Writes video ideas and a script in your character's voice (or paste your own) |
| Voice | Qwen3-TTS VoiceDesign + Base | Builds a voice from a written description once, then reuses it for every script |
| Movement | Wan 2.2 TI2V-5B | Turns your start photo into short clips of the character breathing, blinking, moving |
| Lip-sync | LatentSync 1.5 | Redraws the mouth to match the voice |
| Finishing | Remotion (ffmpeg fallback) | Hook title, word-by-word subtitles, ambient sound, camera drift, loudness |

## How to start
1. Right panel → **Session options → Accelerator: GPU T4 x2** and **Internet: On** (needs phone verification).
2. Set a password in the last cell, then **Run all** (about 5 minutes).
3. Open the **public link** the last cell prints (`https://….gradio.live`), log in as `studio` with your password.
4. Keep this Kaggle tab open while you work: the app runs inside this notebook session and stops with it.

A 35–50 s video takes about 1.5–2 hours on a T4 the first time (mostly video generation and lip-sync).
Kaggle gives about 30 GPU hours a week. Everything you make is saved under `/kaggle/working/studio`
(the Output panel); download your **character pack** so you can reuse the same voice in a new session.

## Build your character first (about 1 hour, once)

You do this once per character. Use any chatbot (Claude, ChatGPT, Gemini) for the text and any image tool
for the pictures. Copy each prompt and fill in the **[brackets]**.

### 1. Character profile
> Create a character profile for an AI influencer in the **[niche]** niche, for **[audience]**. Give:
> a name; age and nationality; a believable backstory (their job, the turning point, why they post);
> a distinct look that is recognisable in a thumbnail (face, hair, 2–3 signature details, clothes,
> colour palette); 3–4 everyday places they film in; a unique selling point (what they do that others
> in the niche don't); voice and tone; a catchphrase; and hard limits (what they never claim or say).
> They are openly AI-generated.

You'll copy parts of the result into the **Character** tab.
Tip: make them ordinary and specific (a real job, a real place, one flaw). Ordinary reads as human.

### 2. Character sheet (reference images)
> Photoreal character reference sheet of **[the look from the profile]**. Three panels on a plain white
> seamless background: full body standing relaxed; side profile, head and shoulders; front close-up
> portrait. The same person and outfit in every panel. Soft natural light, real skin texture with pores
> and small imperfections, not airbrushed, no text.

Generate until the face is one you'd believe. From now on, that face is the character.

### 3. Locations
> Empty location photograph, no people: **[place, e.g. small farmhouse kitchen]**, **[light, e.g. warm
> late-afternoon sun]**, taken on a phone at eye level, vertical 9:16, lived-in and real, no text, no brands.

Make one for each place in the profile.

### 4. The start frame (the photo the studio animates)
In an image tool that accepts two reference images:
> Put the person from image 1 into the place in image 2. Waist-up, facing the camera, relaxed, as if
> about to talk to a friend. Phone-camera look, vertical 9:16. Lit by the room's own light. Eyes looking
> into the lens, mouth closed and relaxed.

**The most important rule: the face must be big.** It should fill at least a quarter of the frame width, so
frame waist-up or closer, never full body. The video model works at 480×832, and a small face comes out with
smeared eyes.

### 5. Voice description
One paragraph describing how they sound:
> **[Gender]**, **[age]**, **[accent]**. **[Pitch]**, **[texture]** (e.g. a faint lived-in rasp, not radio
> polish). **[Pace]** with natural pauses. **[Personality in the delivery]** (e.g. dry warmth, never salesy).
> **[Recording style]** (e.g. intimate close-mic podcast).

### 6. Make videos
Work through the tabs from left to right:
1. **Character:** fill it in, upload the start frame, save.
2. **Script:** get ideas, write the script, then **Use this script**.
3. **Voice:** design the voice once, then speak the script.
4. **Video:** make the clips.
5. **Lip-sync & finish:** sync the mouth, then add the title, subtitles and sound.

After the first video, **Autopilot** on the Script tab runs steps 3 to 5 in one go.
**Export character pack** saves the voice, so the character sounds the same in every video.

### Rules worth keeping
- **Say it's AI.** TikTok, Instagram and YouTube all require realistic AI-generated content to be labelled,
  so use their AI-content toggle. Never present the character as a real person or copy a real person's face.
- **Stay inside your niche's limits.** For health and money especially: no diagnosis, no guarantees, no
  invented studies. The script writer is told this, but you check every script before it's spoken.
- **Licences.** Check each model's licence before earning money with it (see the README). Remotion is free for
  individuals and companies of up to 3 people; larger companies need a licence.

## Setup

In [ ]:
# The studio's code, embedded so this notebook needs nothing else. Unpacked to /tmp/studio.
FILES = {
"remotion/package.json": "{\n  \"name\": \"influencer-studio-finish\",\n  \"version\": \"1.0.0\",\n  \"private\": true,\n  \"description\": \"Finishing pass for AI Influencer Studio: subtitles, hook title, ambient bed, camera drift.\",\n  \"scripts\": {\n    \"audio\": \"node tools/make-audio.mjs\",\n    \"studio\": \"remotion studio src/index.ts\",\n    \"render\": \"remotion render src/index.ts Talking out/final.mp4 --props=props.json --codec=h264 --crf=18\"\n  },\n  \"dependencies\": {\n    \"@remotion/cli\": \"4.0.486\",\n    \"react\": \"18.3.1\",\n    \"react-dom\": \"18.3.1\",\n    \"remotion\": \"4.0.486\"\n  },\n  \"devDependencies\": {\n    \"@types/react\": \"18.3.12\",\n    \"typescript\": \"5.5.4\"\n  }\n}\n",
"remotion/remotion.config.ts": "import {Config} from '@remotion/cli/config';\n\nConfig.setVideoImageFormat('jpeg');\nConfig.setOverwriteOutput(true);\nConfig.setConcurrency(null); // auto: use all available cores\n",
"remotion/src/Root.tsx": "import React from 'react';\nimport {Composition} from 'remotion';\nimport {FPS, Talking, TalkingProps} from './Talking';\n\nconst defaults: TalkingProps = {\n  video: 'video/talking.mp4',\n  voice: 'audio/voice.wav',\n  lines: [],\n  hook: '',\n  durationSeconds: 10,\n  ambientVolume: 0.18,\n};\n\nexport const Root: React.FC = () => (\n  <Composition\n    id=\"Talking\"\n    component={Talking}\n    fps={FPS}\n    width={1080}\n    height={1920}\n    durationInFrames={10 * FPS}\n    defaultProps={defaults}\n    // The real length comes from the voice-over, passed in with --props.\n    calculateMetadata={({props}) => ({durationInFrames: Math.ceil(props.durationSeconds * FPS)})}\n  />\n);\n",
"remotion/src/Talking.tsx": "import React from 'react';\nimport {\n  AbsoluteFill,\n  Audio,\n  OffthreadVideo,\n  interpolate,\n  spring,\n  staticFile,\n  useCurrentFrame,\n  useVideoConfig,\n} from 'remotion';\n\nexport const FPS = 30;\n\nexport type CaptionLine = {text: string; from: number; to: number}; // seconds\n\nexport type TalkingProps = {\n  video: string; // lip-synced clip, relative to public/\n  voice: string; // clean voice-over, relative to public/\n  lines: CaptionLine[];\n  hook: string; // on-screen title for the first seconds; '' for none\n  durationSeconds: number;\n  ambientVolume: number; // 0 = no ambient bed\n};\n\nconst clamp = {extrapolateLeft: 'clamp', extrapolateRight: 'clamp'} as const;\nconst FONT = '\"Helvetica Neue\", Helvetica, Arial, \"DejaVu Sans\", sans-serif';\n\n// Slow push-in plus a slight handheld sway, as if the propped phone breathes.\nconst Camera: React.FC<{children: React.ReactNode}> = ({children}) => {\n  const f = useCurrentFrame();\n  const {durationInFrames} = useVideoConfig();\n  const t = f / durationInFrames;\n  const scale = interpolate(t, [0, 1], [1.02, 1.08]);\n  const x = 4 * Math.sin(t * Math.PI * 2 * 0.7);\n  const y = 3 * Math.sin(t * Math.PI * 2 * 1.1 + 1);\n  const rot = 0.25 * Math.sin(t * Math.PI * 2 * 0.5);\n  return (\n    <AbsoluteFill\n      style={{transform: `translate(${x}px, ${y}px) rotate(${rot}deg) scale(${scale})`, transformOrigin: '50% 40%'}}\n    >\n      {children}\n    </AbsoluteFill>\n  );\n};\n\n// One subtitle line: words pop in one after another, the line fades out at its end.\n// A dark box behind it keeps it readable over light clothes and bright backgrounds.\nconst Line: React.FC<{line: CaptionLine}> = ({line}) => {\n  const f = useCurrentFrame();\n  const {fps} = useVideoConfig();\n  const start = line.from * fps;\n  const end = line.to * fps;\n  if (f < start - 1 || f > end + 6) return null;\n\n  const words = line.text.split(' ');\n  const perWord = Math.min(5, ((end - start) * 0.45) / words.length);\n  const out = interpolate(f, [end - 4, end + 4], [1, 0], clamp);\n\n  return (\n    <div\n      style={{\n        alignSelf: 'center',\n        maxWidth: 920,\n        display: 'flex',\n        flexWrap: 'wrap',\n        justifyContent: 'center',\n        gap: '0 16px',\n        padding: '16px 32px 20px',\n        borderRadius: 22,\n        backgroundColor: 'rgba(12,16,10,0.62)',\n        opacity: out,\n      }}\n    >\n      {words.map((w, i) => {\n        const s = spring({frame: f - start - i * perWord, fps, config: {damping: 16, stiffness: 180}});\n        return (\n          <span\n            key={i}\n            style={{\n              display: 'inline-block',\n              opacity: s,\n              transform: `translateY(${(1 - s) * 18}px)`,\n              color: '#fff',\n              fontFamily: FONT,\n              fontWeight: 700,\n              fontSize: 60,\n              lineHeight: 1.2,\n              letterSpacing: -0.5,\n            }}\n          >\n            {w}\n          </span>\n        );\n      })}\n    </div>\n  );\n};\n\n// The hook title at the top for the first few seconds.\nconst Hook: React.FC<{text: string}> = ({text}) => {\n  const f = useCurrentFrame();\n  const {fps} = useVideoConfig();\n  const s = spring({frame: f - 4, fps, config: {damping: 15, stiffness: 150}});\n  const out = interpolate(f, [3.6 * fps, 4.2 * fps], [1, 0], clamp);\n  if (!text || f > 4.2 * fps) return null;\n  return (\n    <AbsoluteFill style={{alignItems: 'center', paddingTop: 210}}>\n      <div\n        style={{\n          maxWidth: 900,\n          padding: '22px 36px',\n          borderRadius: 26,\n          backgroundColor: '#fff',\n          color: '#111',\n          fontFamily: FONT,\n          fontWeight: 800,\n          fontSize: 64,\n          lineHeight: 1.12,\n          textAlign: 'center',\n          opacity: s * out,\n          transform: `scale(${0.85 + s * 0.15}) rotate(-1.5deg)`,\n        }}\n      >\n        {text}\n      </div>\n    </AbsoluteFill>\n  );\n};\n\nexport const Talking: React.FC<TalkingProps> = ({video, voice, lines, hook, ambientVolume}) => {\n  const {durationInFrames} = useVideoConfig();\n  return (\n    <AbsoluteFill style={{backgroundColor: '#000'}}>\n      <Camera>\n        <OffthreadVideo src={staticFile(video)} muted style={{width: '100%', height: '100%', objectFit: 'cover'}} />\n      </Camera>\n\n      <AbsoluteFill\n        style={{background: 'linear-gradient(0deg, rgba(0,0,0,0.45) 0%, rgba(0,0,0,0) 32%)', pointerEvents: 'none'}}\n      />\n\n      <Hook text={hook} />\n\n      <AbsoluteFill style={{justifyContent: 'flex-end', paddingBottom: 300}}>\n        {lines.map((l, i) => (\n          <Line key={i} line={l} />\n        ))}\n      </AbsoluteFill>\n\n      <Audio src={staticFile(voice)} />\n      {ambientVolume > 0 && (\n        <Audio\n          src={staticFile('audio/ambient.wav')}\n          loop\n          volume={(fr) =>\n            ambientVolume *\n            interpolate(fr, [0, 8], [0, 1], clamp) *\n            interpolate(fr, [durationInFrames - 10, durationInFrames], [1, 0], clamp)\n          }\n        />\n      )}\n    </AbsoluteFill>\n  );\n};\n",
"remotion/src/index.ts": "import {registerRoot} from 'remotion';\nimport {Root} from './Root';\n\nregisterRoot(Root);\n",
"remotion/tools/make-audio.mjs": "// Procedural ambient bed (garden at dusk) \u2014 no external deps.\n// Faint afternoon birdsong + soft breeze through leaves, no voice, no music.\n// Writes a 16-bit mono WAV into public/audio/. The ends are crossfaded so the\n// bed loops cleanly with the video.\n//\n//   node tools/make-audio.mjs\n//\nimport {mkdirSync, writeFileSync} from 'node:fs';\nimport {fileURLToPath} from 'node:url';\nimport {dirname, join} from 'node:path';\n\nconst SR = 44100;\nconst SECONDS = 10;\nconst OUT = join(dirname(fileURLToPath(import.meta.url)), '..', 'public', 'audio');\nmkdirSync(OUT, {recursive: true});\n\nconst TAU = Math.PI * 2;\n\n// Deterministic PRNG so every run produces the same bed.\nlet seed = 1337;\nconst rand = () => {\n  seed = (seed * 1664525 + 1013904223) >>> 0;\n  return seed / 4294967296;\n};\n\n// Render a little longer than needed so the tail can be folded onto the head.\nconst FOLD = 0.6;\nconst N = Math.ceil((SECONDS + FOLD) * SR);\nconst buf = new Float32Array(N);\n\n// ----- breeze: low-passed noise with slow gusting ------------------------------\n{\n  let lp1 = 0;\n  let lp2 = 0;\n  for (let i = 0; i < N; i++) {\n    const t = i / SR;\n    const gust =\n      0.55 +\n      0.25 * Math.sin(TAU * 0.13 * t + 0.7) +\n      0.15 * Math.sin(TAU * 0.31 * t + 2.1);\n    const white = rand() * 2 - 1;\n    lp1 += 0.035 * (white - lp1); // body of the wind\n    lp2 += 0.25 * (white - lp2); // leaf rustle (brighter)\n    const rustle = (lp2 - lp1) * (0.4 + 0.6 * Math.max(0, Math.sin(TAU * 0.21 * t)));\n    buf[i] += (lp1 * 1.6 + rustle * 0.25) * gust * 0.22;\n  }\n}\n\n// ----- birdsong: short swept chirps in small phrases ---------------------------\nfunction chirp(t0, dur, f0, f1, gain) {\n  const n0 = Math.floor(t0 * SR);\n  const len = Math.floor(dur * SR);\n  let phase = 0;\n  for (let i = 0; i < len; i++) {\n    const idx = n0 + i;\n    if (idx >= N) break;\n    const u = i / len;\n    const f = f0 + (f1 - f0) * Math.sin(u * Math.PI * 0.5) + 180 * Math.sin(TAU * 38 * u * dur);\n    phase += (TAU * f) / SR;\n    const env = Math.sin(Math.PI * u) ** 2;\n    buf[idx] += (Math.sin(phase) + 0.18 * Math.sin(2 * phase)) * env * gain;\n  }\n}\n\n// Two \"birds\" at different distances, phrases scattered through the take.\nconst birds = [\n  {base: 3400, spread: 900, gain: 0.05, every: 2.3},\n  {base: 4600, spread: 1200, gain: 0.028, every: 3.1},\n];\nfor (const b of birds) {\n  let t = rand() * 0.8 + 0.3;\n  while (t < SECONDS + FOLD - 0.5) {\n    const notes = 2 + Math.floor(rand() * 4);\n    for (let k = 0; k < notes; k++) {\n      const f0 = b.base + (rand() - 0.5) * b.spread;\n      const f1 = f0 + (rand() - 0.3) * b.spread;\n      chirp(t, 0.06 + rand() * 0.09, f0, f1, b.gain * (0.7 + rand() * 0.3));\n      t += 0.1 + rand() * 0.12;\n    }\n    t += b.every * (0.6 + rand() * 0.8);\n  }\n}\n\n// ----- fold tail onto head for a seamless loop ---------------------------------\nconst L = Math.floor(SECONDS * SR);\nconst F = Math.floor(FOLD * SR);\nconst out = new Float32Array(L);\nfor (let i = 0; i < L; i++) out[i] = buf[i];\nfor (let i = 0; i < F; i++) {\n  const w = i / F; // head fades in as the folded tail fades out\n  out[i] = buf[i] * w + buf[L + i] * (1 - w);\n}\n\n// Normalise quietly \u2014 this is a ducked ambient bed.\nlet peak = 0;\nfor (const s of out) peak = Math.max(peak, Math.abs(s));\nconst scale = 0.35 / (peak || 1);\n\nconst data = Buffer.alloc(L * 2);\nfor (let i = 0; i < L; i++) {\n  data.writeInt16LE(Math.round(Math.max(-1, Math.min(1, out[i] * scale)) * 32767), i * 2);\n}\nconst header = Buffer.alloc(44);\nheader.write('RIFF', 0);\nheader.writeUInt32LE(36 + data.length, 4);\nheader.write('WAVE', 8);\nheader.write('fmt ', 12);\nheader.writeUInt32LE(16, 16);\nheader.writeUInt16LE(1, 20);\nheader.writeUInt16LE(1, 22);\nheader.writeUInt32LE(SR, 24);\nheader.writeUInt32LE(SR * 2, 28);\nheader.writeUInt16LE(2, 32);\nheader.writeUInt16LE(16, 34);\nheader.write('data', 36);\nheader.writeUInt32LE(data.length, 40);\nwriteFileSync(join(OUT, 'ambient.wav'), Buffer.concat([header, data]));\nconsole.log('wrote public/audio/ambient.wav');\n",
"remotion/tsconfig.json": "{\n  \"compilerOptions\": {\n    \"target\": \"ES2020\",\n    \"module\": \"ESNext\",\n    \"moduleResolution\": \"bundler\",\n    \"jsx\": \"react-jsx\",\n    \"strict\": true,\n    \"noEmit\": true,\n    \"esModuleInterop\": true,\n    \"skipLibCheck\": true,\n    \"lib\": [\"DOM\", \"DOM.Iterable\", \"ES2020\"]\n  },\n  \"include\": [\"src\", \"remotion.config.ts\"]\n}\n",
"studio/app.py": "\"\"\"AI Influencer Studio web UI (Gradio). Each tab is one step; the steps themselves live in pipeline.py.\n\n  python app.py               # local\n  launch(share=True, password=\"...\")   # from the notebook: a public link, protected by a password\n\"\"\"\n\nimport html\nimport inspect\nimport json\nimport time\nfrom pathlib import Path\n\nimport gradio as gr\n\nimport pipeline as P\n\nGUIDE = (Path(__file__).parent / \"guide.md\").read_text()\nFIELDS = [\"name\", \"niche\", \"audience\", \"backstory\", \"usp\", \"tone\", \"catchphrase\", \"hard_limits\", \"look\", \"setting\", \"light\"]\nLABELS = {\n    \"name\": (\"Name\", 1, \"Ada Brooks\"),\n    \"niche\": (\"Niche\", 1, \"mobility and back pain for desk workers\"),\n    \"audience\": (\"Audience\", 1, \"people with desk jobs, sore backs and no time\"),\n    \"backstory\": (\"Backstory\", 4, \"Former NHS physiotherapist, 10 years on hospital wards, burned out in a desk job \"\n                  \"herself, rebuilt with five-minute routines between meetings.\"),\n    \"usp\": (\"What makes them different\", 2, \"One small, evidence-based move per video; says plainly when the evidence is thin.\"),\n    \"tone\": (\"Tone of voice\", 1, \"warm, practical, a little dry; talks with you, not at you\"),\n    \"catchphrase\": (\"Catchphrase\", 1, \"Small moves, every day.\"),\n    \"hard_limits\": (\"Hard limits\", 2, \"no diagnosing, no cure claims, no invented studies; sends anything serious to a professional\"),\n    \"look\": (\"Look in the start frame (for the video prompt)\", 2, \"a woman in her late 30s with shoulder-length dark hair \"\n             \"tied back, light freckles, small silver hoops, wearing a soft grey crew-neck jumper\"),\n    \"setting\": (\"Setting in the start frame\", 1, \"a bright small kitchen with plants on the windowsill\"),\n    \"light\": (\"Light\", 1, \"soft daylight from a window on the left, exposure constant\"),\n}\nDEFAULT_VOICE = (\n    \"Warm, grounded British woman, late 30s, neutral modern English accent. Medium pitch, natural chest resonance, \"\n    \"faint lived-in texture, not radio polish. Calm, unhurried pace with natural pauses. Plain-spoken and \"\n    \"reassuring, dry warmth, never salesy or theatrical. Intimate close-mic podcast delivery.\"\n)\nLANGUAGES = [\"English\", \"Chinese\", \"Japanese\", \"Korean\", \"German\", \"French\", \"Russian\", \"Portuguese\", \"Spanish\", \"Italian\"]\nMAX_TAKES = 4\n\n\nCSS = \"\"\"\n.status {display: flex; align-items: center; gap: 14px; padding: 14px 18px; border-radius: 12px;\n          font-size: 15px; line-height: 1.4; border: 1px solid var(--border-color-primary);\n          background: var(--background-fill-secondary);}\n.status b {white-space: nowrap;}\n.status .detail {opacity: .75; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; flex: 1;}\n.status .clock {font-variant-numeric: tabular-nums; opacity: .75;}\n.spinner {width: 20px; height: 20px; flex: none; border-radius: 50%; border: 3px solid rgba(128, 128, 128, .3);\n           border-top-color: var(--color-accent, #f97316); animation: spin .9s linear infinite;}\n@keyframes spin {to {transform: rotate(360deg);}}\n.status.done {border-color: #22c55e;} .status.fail {border-color: #ef4444;}\n\"\"\"\n\n\ndef status_html(state, title, detail=\"\", seconds=0):\n    \"\"\"The bar above a step's output: a spinner while it runs, then a tick or a cross.\"\"\"\n    icon = {\"busy\": '<div class=\"spinner\"></div>', \"done\": \"\u2705\", \"fail\": \"\u274c\"}[state]\n    clock = f'<span class=\"clock\">{int(seconds) // 60}:{int(seconds) % 60:02d}</span>' if seconds else \"\"\n    return (f'<div class=\"status {state}\">{icon}<b>{html.escape(title)}</b>'\n            f'<span class=\"detail\">{html.escape(detail)}</span>{clock}</div>')\n\n\ndef with_status(title, fn):\n    \"\"\"Wrap a guarded step whose first output is its log: adds a live status bar as the last output.\"\"\"\n    def run(*args):\n        t0, failed, out = time.time(), False, None\n        yield (*[gr.update()] * n_outputs[fn], status_html(\"busy\", title, \"starting...\", 0.1))\n        for out in fn(*args):\n            log = out[0] or \"\"\n            failed = \"\u274c FAILED\" in log\n            last = next((l for l in reversed(log.splitlines()) if l.strip()), \"\")\n            state = \"fail\" if failed else \"busy\"\n            yield (*out, status_html(state, title + (\" failed\" if failed else \"...\"), last, time.time() - t0))\n        if out is not None and not failed:\n            yield (*out, status_html(\"done\", title + \" done\", \"\", time.time() - t0))\n    return run\n\n\nn_outputs = {}\n\n\ndef guarded(fn):\n    \"\"\"Show a step's real error message in the UI (Gradio otherwise shows only \"Error\").\"\"\"\n    def message(e):\n        return f\"{type(e).__name__}: {e}\" if str(e) else type(e).__name__\n\n    if inspect.isgeneratorfunction(fn):\n        # Long steps yield (log, ...). On failure, keep the log on screen and add the error to it, so the\n        # user sees what went wrong instead of an \"Error\" badge over an empty box.\n        def run(*args):\n            last = None\n            try:\n                for last in fn(*args):\n                    yield last\n            except gr.Error:\n                raise\n            except Exception as e:  # noqa: BLE001\n                if last is None:\n                    raise gr.Error(message(e)) from e\n                log = (last[0] or \"\") + f\"\\n\\n\u274c FAILED: {message(e)}\"\n                gr.Warning(\"This step failed; the log shows why.\")\n                yield (log, *[gr.update() for _ in last[1:]])\n    else:\n        def run(*args):\n            try:\n                return fn(*args)\n            except gr.Error:\n                raise\n            except Exception as e:  # noqa: BLE001\n                raise gr.Error(message(e)) from e\n    return run\n\n\ndef build():\n    with gr.Blocks(title=\"AI Influencer Studio\") as demo:\n        video_dir = gr.State(\"\")\n        gr.Markdown(\"# AI Influencer Studio\\nCharacter \u2192 idea \u2192 script \u2192 voice \u2192 video \u2192 lip-sync \u2192 finished vertical video. \"\n                    \"Free GPU, open models, no API keys. New here? Start with the **Guide** tab.\")\n\n        # ---------------------------------------------------------------- guide\n        with gr.Tab(\"Guide\"):\n            gr.Markdown(GUIDE)\n\n        # ---------------------------------------------------------------- character\n        with gr.Tab(\"1 \u00b7 Character\"):\n            with gr.Row():\n                saved = gr.Dropdown(P.list_characters(), label=\"Saved characters\", scale=3)\n                load_btn = gr.Button(\"Load\", scale=1)\n                pack_in = gr.File(label=\"Import character pack (.zip)\", file_types=[\".zip\"], scale=2, height=120)\n            with gr.Row():\n                with gr.Column(scale=3):\n                    boxes = {f: gr.Textbox(label=LABELS[f][0], lines=LABELS[f][1], value=LABELS[f][2]) for f in FIELDS}\n                with gr.Column(scale=2):\n                    start = gr.Image(label=\"Start frame (waist-up, face big, 9:16)\", type=\"filepath\", height=520)\n                    save_btn = gr.Button(\"Save character\", variant=\"primary\")\n                    char_status = gr.Markdown()\n                    export_btn = gr.Button(\"Export character pack\")\n                    pack_out = gr.File(label=\"Character pack\")\n\n        # ---------------------------------------------------------------- script\n        with gr.Tab(\"2 \u00b7 Script\"):\n            gr.Markdown(\"Paste your own script, **or** get ideas and let the writer draft one. Blank line between beats. \"\n                        \"Read it before you use it: you're responsible for what the character says.\")\n            with gr.Row():\n                theme = gr.Textbox(label=\"Theme (optional)\", placeholder=\"e.g. sleep, common myths\", scale=3)\n                ideas_btn = gr.Button(\"Suggest ideas\", scale=1)\n            idea_pick = gr.Radio([], label=\"Ideas\")\n            with gr.Row():\n                seconds = gr.Slider(15, 60, value=35, step=5, label=\"Length (seconds)\")\n                write_btn = gr.Button(\"Write script for this idea\")\n            script = gr.Textbox(label=\"Script\", lines=10)\n            hook = gr.Textbox(label=\"Hook title (on screen for the first 4 s; empty for none)\")\n            with gr.Row():\n                use_btn = gr.Button(\"Use this script\", variant=\"primary\")\n                auto_btn = gr.Button(\"Autopilot: voice \u2192 video \u2192 lip-sync \u2192 finish\")\n            script_status = gr.Markdown()\n            auto_status = gr.HTML()\n            auto_log = gr.Textbox(label=\"Autopilot log\", lines=10, autoscroll=True)\n            auto_out = gr.Video(label=\"Final video\", height=560)\n\n        # ---------------------------------------------------------------- voice\n        with gr.Tab(\"3 \u00b7 Voice\"):\n            gr.Markdown(\"**First time:** describe the voice, design a few takes, pick one, **Use this voice**. \"\n                        \"It's saved with the character and reused for every script, so they always sound the same.\")\n            current_voice = gr.Audio(label=\"Saved voice\", type=\"filepath\", interactive=False)\n            voice_desc = gr.Textbox(label=\"Voice description\", lines=4, value=DEFAULT_VOICE)\n            with gr.Row():\n                language = gr.Dropdown(LANGUAGES, value=\"English\", label=\"Language\")\n                n_takes = gr.Slider(2, MAX_TAKES, value=4, step=1, label=\"Takes\")\n                design_btn = gr.Button(\"Design voice\")\n            voice_status = gr.HTML()\n            take_audio = [gr.Audio(label=f\"Take {i + 1}\", type=\"filepath\", interactive=False, visible=False)\n                          for i in range(MAX_TAKES)]\n            with gr.Row():\n                take_pick = gr.Radio([], label=\"Pick a take\")\n                lock_btn = gr.Button(\"Use this voice\", variant=\"primary\")\n            speak_btn = gr.Button(\"Speak the script\", variant=\"primary\")\n            voice_log = gr.Textbox(label=\"Log\", lines=8, autoscroll=True)\n            voice_out = gr.Audio(label=\"Voice-over\", type=\"filepath\", interactive=False)\n\n        # ---------------------------------------------------------------- video\n        with gr.Tab(\"4 \u00b7 Video\"):\n            gr.Markdown(\"Each seed is one ~5 s take of the character facing the camera (about 20 min each on a T4). \"\n                        \"Lip-sync adds the speaking later, so the mouth stays relaxed here.\")\n            prompt = gr.Textbox(label=\"Prompt\", lines=5)\n            negative = gr.Textbox(label=\"Negative prompt\", lines=3, value=P.DEFAULT_NEGATIVE)\n            with gr.Row():\n                seeds = gr.Textbox(label=\"Seeds (one take each)\", value=\"42, 7, 1234\")\n                steps = gr.Slider(20, 50, value=40, step=5, label=\"Steps (fewer = faster, softer)\")\n                takes_btn = gr.Button(\"Make takes\", variant=\"primary\")\n            takes_status = gr.HTML()\n            takes_log = gr.Textbox(label=\"Log\", lines=8, autoscroll=True)\n            with gr.Row():\n                take_videos = [gr.Video(label=f\"Take {i + 1}\", height=360, visible=False) for i in range(MAX_TAKES)]\n\n        # ---------------------------------------------------------------- lip-sync & finish\n        with gr.Tab(\"5 \u00b7 Lip-sync & finish\"):\n            use_takes = gr.CheckboxGroup([], label=\"Takes to use (in this order); skip any where the face warps\")\n            with gr.Row():\n                keep = gr.Number(value=0, minimum=0, maximum=5, step=0.5, label=\"Keep only the first N seconds of each take (0 = all)\")\n                fp16 = gr.Checkbox(label=\"Faster lip-sync (fp16); turn off if the mouth looks noisy\")\n                ls_btn = gr.Button(\"Lip-sync\", variant=\"primary\")\n            ls_status = gr.HTML()\n            ls_log = gr.Textbox(label=\"Log\", lines=8, autoscroll=True)\n            ls_out = gr.Video(label=\"Lip-synced\", height=480)\n            with gr.Row():\n                fin_hook = gr.Textbox(label=\"Hook title\")\n                ambient = gr.Slider(0, 0.5, value=0.18, step=0.02, label=\"Ambient sound\")\n                engine = gr.Radio([\"remotion\", \"ffmpeg\"], value=\"remotion\", label=\"Finish with\")\n                fin_btn = gr.Button(\"Finish\", variant=\"primary\")\n            fin_status = gr.HTML()\n            fin_log = gr.Textbox(label=\"Log\", lines=6, autoscroll=True)\n            final = gr.Video(label=\"Final video\", height=640)\n            final_file = gr.File(label=\"Download\")\n\n        # ================================================================ wiring\n        def fields_of(*vals):\n            f = dict(zip(FIELDS, vals))\n            if not f[\"name\"].strip():\n                raise gr.Error(\"Give the character a name first.\")\n            return f\n\n        def voice_state(name):\n            ref, meta = P.saved_voice(name) if name.strip() else (None, None)\n            return ref, (meta or {}).get(\"description\", DEFAULT_VOICE), (meta or {}).get(\"language\", \"English\")\n\n        @guarded\n        def on_save(start_img, *vals):\n            f = fields_of(*vals)\n            d = P.save_character(f, start_img)\n            has_start = (d / \"start.png\").exists()\n            msg = f\"Saved **{f['name']}**.\" + (\"\" if has_start else \" \u26a0 Upload a start frame before making video.\")\n            return msg, gr.update(choices=P.list_characters(), value=d.name), P.video_prompt(f), *voice_state(f[\"name\"])\n\n        @guarded\n        def on_load(char_slug):\n            if not char_slug:\n                raise gr.Error(\"Pick a saved character.\")\n            f, start_img = P.load_character(char_slug)\n            return (*[f.get(k, \"\") for k in FIELDS], start_img, P.video_prompt(f), *voice_state(f[\"name\"]))\n\n        @guarded\n        def on_import(zip_file):\n            if not zip_file:\n                return gr.update()\n            return gr.update(choices=P.list_characters(), value=P.import_pack(zip_file))\n\n        save_btn.click(on_save, [start, *boxes.values()], [char_status, saved, prompt, current_voice, voice_desc, language])\n        load_btn.click(on_load, [saved], [*boxes.values(), start, prompt, current_voice, voice_desc, language])\n        pack_in.upload(on_import, [pack_in], [saved])\n        export_btn.click(guarded(P.export_pack), [boxes[\"name\"]], [pack_out])\n\n        @guarded\n        def on_ideas(theme_, *vals):\n            found = P.ideas(fields_of(*vals), theme_)\n            return gr.update(choices=found, value=found[0] if found else None)\n\n        @guarded\n        def on_write(idea, secs, *vals):\n            if not idea:\n                raise gr.Error(\"Pick an idea first (or paste your own script).\")\n            return P.write_script(fields_of(*vals), idea, int(secs))\n\n        @guarded\n        def on_use(script_, hook_, *vals):\n            f = fields_of(*vals)\n            if not script_.strip():\n                raise gr.Error(\"The script is empty.\")\n            d = P.new_video(f[\"name\"], script_, hook_)\n            return d, f\"Script saved for **{f['name']}**. Next: the **Voice** tab.\", hook_\n\n        ideas_btn.click(on_ideas, [theme, *boxes.values()], [idea_pick])\n        write_btn.click(on_write, [idea_pick, seconds, *boxes.values()], [script, hook])\n        use_btn.click(on_use, [script, hook, *boxes.values()], [video_dir, script_status, fin_hook])\n\n        @guarded\n        def on_design(name, desc, lang, n):\n            n = int(n)\n            for log, paths in P.design_voice(name, desc, lang, n):\n                audios = [gr.update(value=paths[i] if i < len(paths) else None, visible=i < n) for i in range(MAX_TAKES)]\n                pick = gr.update(choices=[(f\"Take {i + 1}\", p) for i, p in enumerate(paths)],\n                                 value=paths[0] if paths else None)\n                yield log, *audios, pick\n\n        @guarded\n        def on_lock(name, take, desc, lang):\n            if not take:\n                raise gr.Error(\"Design the voice and pick a take first.\")\n            return P.lock_voice(name, take, desc, lang)\n\n        @guarded\n        def on_speak(name, vd):\n            if not vd:\n                raise gr.Error(\"Save a script first (Script tab \u2192 Use this script).\")\n            for log, path in P.speak(name, vd):\n                yield log, path\n\n        n_outputs[on_design] = len([voice_log, *take_audio, take_pick])\n        design_btn.click(with_status(\"Designing the voice\", on_design), [boxes[\"name\"], voice_desc, language, n_takes], [*[voice_log, *take_audio, take_pick], voice_status])\n        lock_btn.click(on_lock, [boxes[\"name\"], take_pick, voice_desc, language], [current_voice])\n        n_outputs[on_speak] = len([voice_log, voice_out])\n        speak_btn.click(with_status(\"Speaking the script\", on_speak), [boxes[\"name\"], video_dir], [*[voice_log, voice_out], voice_status])\n\n        def parse_seeds(text):\n            try:\n                s = [int(x) for x in text.replace(\",\", \" \").split()]\n            except ValueError:\n                raise gr.Error(\"Seeds must be whole numbers, e.g. 42, 7, 1234.\") from None\n            if not 1 <= len(s) <= MAX_TAKES:\n                raise gr.Error(f\"Use 1 to {MAX_TAKES} seeds.\")\n            return s\n\n        def take_updates(paths):\n            vids = [gr.update(value=paths[i] if i < len(paths) else None, visible=i < len(paths)) for i in range(MAX_TAKES)]\n            choices = [(Path(p).stem.replace(\"take_\", \"\"), p) for p in paths]\n            return vids, gr.update(choices=choices, value=[p for _, p in choices])\n\n        @guarded\n        def on_takes(name, vd, prompt_, negative_, seeds_, steps_):\n            if not vd:\n                raise gr.Error(\"Save a script first (Script tab \u2192 Use this script).\")\n            for log, paths in P.make_takes(name, vd, prompt_, negative_, parse_seeds(seeds_), int(steps_)):\n                vids, choices = take_updates(paths)\n                yield log, *vids, choices\n\n        n_outputs[on_takes] = len([takes_log, *take_videos, use_takes])\n        takes_btn.click(with_status(\"Making the video takes\", on_takes), [boxes[\"name\"], video_dir, prompt, negative, seeds, steps], [*[takes_log, *take_videos, use_takes], takes_status])\n\n        @guarded\n        def on_lipsync(vd, takes, keep_, fp16_):\n            if not vd:\n                raise gr.Error(\"Save a script first.\")\n            for log, path in P.lipsync(vd, takes, float(keep_ or 0), fp16_):\n                yield log, path\n\n        @guarded\n        def on_finish(vd, hook_, amb, eng):\n            if not vd or not (Path(vd) / \"lipsync.mp4\").exists():\n                raise gr.Error(\"Lip-sync first.\")\n            for log, path in P.finish(vd, hook_, float(amb), eng):\n                yield log, path, path\n\n        n_outputs[on_lipsync] = len([ls_log, ls_out])\n        ls_btn.click(with_status(\"Lip-syncing\", on_lipsync), [video_dir, use_takes, keep, fp16], [*[ls_log, ls_out], ls_status])\n        n_outputs[on_finish] = len([fin_log, final, final_file])\n        fin_btn.click(with_status(\"Finishing the video\", on_finish), [video_dir, fin_hook, ambient, engine], [*[fin_log, final, final_file], fin_status])\n\n        @guarded\n        def on_auto(script_, hook_, prompt_, negative_, seeds_, steps_, amb, *vals):\n            f = fields_of(*vals)\n            if not P.saved_voice(f[\"name\"])[0]:\n                raise gr.Error(\"Autopilot needs a saved voice: design one on the Voice tab first.\")\n            if not script_.strip():\n                raise gr.Error(\"The script is empty.\")\n            vd = P.new_video(f[\"name\"], script_, hook_)\n            history = []\n            for title, gen in (\n                (\"VOICE\", P.speak(f[\"name\"], vd)),\n                (\"VIDEO\", P.make_takes(f[\"name\"], vd, prompt_ or P.video_prompt(f), negative_, parse_seeds(seeds_), int(steps_))),\n            ):\n                for log, _ in gen:\n                    yield \"\\n\".join(history + [f\"== {title} ==\", log]), None, vd\n                history += [f\"== {title} ==\", log.splitlines()[-1] if log else \"\"]\n            takes = sorted(str(p) for p in (Path(vd) / \"takes\").glob(\"take_seed*.mp4\"))\n            for title, gen in ((\"LIP-SYNC\", P.lipsync(vd, takes, 0)), (\"FINISH\", P.finish(vd, hook_, float(amb)))):\n                for log, path in gen:\n                    yield \"\\n\".join(history + [f\"== {title} ==\", log]), path if title == \"FINISH\" else None, vd\n                history += [f\"== {title} ==\", log.splitlines()[-1] if log else \"\"]\n\n        n_outputs[on_auto] = 3\n        auto_btn.click(with_status(\"Autopilot\", on_auto), [script, hook, prompt, negative, seeds, steps, ambient, *boxes.values()],\n                       [auto_log, auto_out, video_dir, auto_status])\n    return demo\n\n\ndef launch(share=True, password=None):\n    demo = build()\n    auth = (\"studio\", password) if password else None\n    demo.queue(default_concurrency_limit=1).launch(\n        share=share, auth=auth, allowed_paths=[str(P.HOME.resolve()), \"/tmp\"], show_error=True, theme=gr.themes.Soft(), css=CSS,\n    )\n    return demo\n\n\nif __name__ == \"__main__\":\n    build().queue(default_concurrency_limit=1).launch(\n        allowed_paths=[str(P.HOME.resolve())], show_error=True, theme=gr.themes.Soft(), css=CSS\n    )\n",
"studio/guide.md": "## Build your character first (about 1 hour, once)\n\nYou do this once per character. Use any chatbot (Claude, ChatGPT, Gemini) for the text and any image tool\nfor the pictures. Copy each prompt and fill in the **[brackets]**.\n\n### 1. Character profile\n> Create a character profile for an AI influencer in the **[niche]** niche, for **[audience]**. Give:\n> a name; age and nationality; a believable backstory (their job, the turning point, why they post);\n> a distinct look that is recognisable in a thumbnail (face, hair, 2\u20133 signature details, clothes,\n> colour palette); 3\u20134 everyday places they film in; a unique selling point (what they do that others\n> in the niche don't); voice and tone; a catchphrase; and hard limits (what they never claim or say).\n> They are openly AI-generated.\n\nYou'll copy parts of the result into the **Character** tab.\nTip: make them ordinary and specific (a real job, a real place, one flaw). Ordinary reads as human.\n\n### 2. Character sheet (reference images)\n> Photoreal character reference sheet of **[the look from the profile]**. Three panels on a plain white\n> seamless background: full body standing relaxed; side profile, head and shoulders; front close-up\n> portrait. The same person and outfit in every panel. Soft natural light, real skin texture with pores\n> and small imperfections, not airbrushed, no text.\n\nGenerate until the face is one you'd believe. From now on, that face is the character.\n\n### 3. Locations\n> Empty location photograph, no people: **[place, e.g. small farmhouse kitchen]**, **[light, e.g. warm\n> late-afternoon sun]**, taken on a phone at eye level, vertical 9:16, lived-in and real, no text, no brands.\n\nMake one for each place in the profile.\n\n### 4. The start frame (the photo the studio animates)\nIn an image tool that accepts two reference images:\n> Put the person from image 1 into the place in image 2. Waist-up, facing the camera, relaxed, as if\n> about to talk to a friend. Phone-camera look, vertical 9:16. Lit by the room's own light. Eyes looking\n> into the lens, mouth closed and relaxed.\n\n**The most important rule: the face must be big.** It should fill at least a quarter of the frame width, so\nframe waist-up or closer, never full body. The video model works at 480\u00d7832, and a small face comes out with\nsmeared eyes.\n\n### 5. Voice description\nOne paragraph describing how they sound:\n> **[Gender]**, **[age]**, **[accent]**. **[Pitch]**, **[texture]** (e.g. a faint lived-in rasp, not radio\n> polish). **[Pace]** with natural pauses. **[Personality in the delivery]** (e.g. dry warmth, never salesy).\n> **[Recording style]** (e.g. intimate close-mic podcast).\n\n### 6. Make videos\nWork through the tabs from left to right:\n1. **Character:** fill it in, upload the start frame, save.\n2. **Script:** get ideas, write the script, then **Use this script**.\n3. **Voice:** design the voice once, then speak the script.\n4. **Video:** make the clips.\n5. **Lip-sync & finish:** sync the mouth, then add the title, subtitles and sound.\n\nAfter the first video, **Autopilot** on the Script tab runs steps 3 to 5 in one go.\n**Export character pack** saves the voice, so the character sounds the same in every video.\n\n### Rules worth keeping\n- **Say it's AI.** TikTok, Instagram and YouTube all require realistic AI-generated content to be labelled,\n  so use their AI-content toggle. Never present the character as a real person or copy a real person's face.\n- **Stay inside your niche's limits.** For health and money especially: no diagnosis, no guarantees, no\n  invented studies. The script writer is told this, but you check every script before it's spoken.\n- **Licences.** Check each model's licence before earning money with it (see the README). Remotion is free for\n  individuals and companies of up to 3 people; larger companies need a licence.\n",
"studio/ideas.py": "\"\"\"Video ideas and scripts from the character profile, with a small open LLM on the GPU (no API key).\"\"\"\n\nimport gc\nimport json\nimport re\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nMODEL = \"Qwen/Qwen3-4B-Instruct-2507\"\n\n\nclass Writer:\n    def __init__(self, character, model=MODEL):\n        self.character = character\n        self.tok = AutoTokenizer.from_pretrained(model)\n        if not torch.cuda.is_available():\n            raise RuntimeError(\"The script writer needs a GPU: turn on GPU T4 x2 in Kaggle's session options.\")\n        self.model = AutoModelForCausalLM.from_pretrained(model, dtype=torch.float16, device_map=\"cuda:0\")\n\n    def _ask(self, prompt, max_new_tokens=900):\n        system = (\n            \"You write short-form vertical video content for this creator. Stay in their voice and inside their \"\n            \"hard limits. Never invent statistics or studies; if a claim is uncertain, say so in the script.\\n\\n\"\n            \"CREATOR PROFILE:\\n\" + json.dumps(self.character, indent=2)\n        )\n        msgs = [{\"role\": \"system\", \"content\": system}, {\"role\": \"user\", \"content\": prompt}]\n        # return_dict: transformers 5 returns a dict here by default (input_ids + attention_mask); ask for it explicitly\n        inputs = self.tok.apply_chat_template(\n            msgs, add_generation_prompt=True, return_tensors=\"pt\", return_dict=True\n        ).to(\"cuda:0\")\n        with torch.no_grad():\n            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.8, top_p=0.9)\n        return self.tok.decode(out[0, inputs[\"input_ids\"].shape[1] :], skip_special_tokens=True).strip()\n\n    def ideas(self, n=8, theme=\"\"):\n        text = self._ask(\n            f\"Give {n} video ideas{' about ' + theme if theme else ''}. One per line, numbered, in the form:\\n\"\n            \"1. <punchy title> | <one-sentence angle>\\nNo other text.\"\n        )\n        return [re.sub(r\"^\\s*\\d+[.)]\\s*\", \"\", l).strip() for l in text.splitlines() if re.match(r\"^\\s*\\d+[.)]\", l)]\n\n    def script(self, idea, seconds=35):\n        words = int(seconds * 2.4)\n        text = self._ask(\n            f\"Write the spoken script for this video: {idea}\\n\\n\"\n            f\"About {words} words ({seconds} seconds). Spoken to camera, like talking to a friend. Short sentences. \"\n            \"Open with a hook in the first sentence, end with a one-line call to follow. Blank line between beats. \"\n            \"Then, on a final line, write: HOOK: <on-screen title, max 7 words>\\nNo stage directions, no emojis.\"\n        )\n        hook = \"\"\n        m = re.search(r\"^\\s*HOOK:\\s*(.+)$\", text, flags=re.M)\n        if m:\n            hook = m.group(1).strip().strip('\"')\n            text = text[: m.start()].strip()\n        return text, hook\n\n    def close(self):\n        del self.model\n        gc.collect()\n        torch.cuda.empty_cache()\n",
"studio/media.py": "\"\"\"ffmpeg steps: stretch base takes under the voice-over, a plain finishing fallback, loudness.\"\"\"\n\nimport subprocess\nimport tempfile\nfrom pathlib import Path\n\n\ndef run(*cmd):\n    subprocess.run([str(c) for c in cmd], check=True)\n\n\ndef duration(path):\n    out = subprocess.run(\n        [\"ffprobe\", \"-v\", \"error\", \"-show_entries\", \"format=duration\", \"-of\", \"csv=p=0\", str(path)],\n        check=True, capture_output=True, text=True,\n    )\n    return float(out.stdout)\n\n\ndef fit_to_audio(voice, clips, out_video, out_wav, keep=None, fps=25, size=(480, 832)):\n    \"\"\"Chain the base takes under the voice-over, ready for lip-sync.\n\n    Each take plays forward then backward (ping-pong), so it ends on its own first frame; takes made from\n    the same start frame share that frame, so chaining them gives variety with no visible jump. The chain\n    repeats as needed and is cut to the voice length. `keep` maps a clip path to the seconds to keep from\n    its start (to drop an unwanted gesture at the end). LatentSync expects 25 fps and 16 kHz mono audio.\n    \"\"\"\n    keep = keep or {}\n    w, h = size\n    with tempfile.TemporaryDirectory() as tmp:\n        tmp = Path(tmp)\n        listing = tmp / \"list.txt\"\n        with listing.open(\"w\") as f:\n            for i, clip in enumerate(clips):\n                trim = f\"trim=end={keep[clip]},\" if clip in keep else \"\"\n                pp = tmp / f\"pp{i}.mp4\"\n                run(\n                    \"ffmpeg\", \"-v\", \"error\", \"-y\", \"-i\", clip, \"-filter_complex\",\n                    f\"[0:v]{trim}fps={fps},scale={w}:{h}:flags=lanczos,setsar=1,split[f][b];\"\n                    \"[b]reverse,trim=start_frame=1,setpts=PTS-STARTPTS[r];[f][r]concat=n=2:v=1:a=0[v]\",\n                    \"-map\", \"[v]\", \"-an\", \"-c:v\", \"libx264\", \"-crf\", \"14\", \"-pix_fmt\", \"yuv420p\", pp,\n                )\n                f.write(f\"file '{pp}'\\n\")\n        chain = tmp / \"chain.mp4\"\n        run(\"ffmpeg\", \"-v\", \"error\", \"-y\", \"-f\", \"concat\", \"-safe\", \"0\", \"-i\", listing, \"-c\", \"copy\", chain)\n        run(\n            \"ffmpeg\", \"-v\", \"error\", \"-y\", \"-stream_loop\", \"-1\", \"-i\", chain, \"-i\", voice,\n            \"-map\", \"0:v\", \"-map\", \"1:a\", \"-c:v\", \"libx264\", \"-crf\", \"14\", \"-pix_fmt\", \"yuv420p\",\n            \"-c:a\", \"aac\", \"-b:a\", \"192k\", \"-t\", f\"{duration(voice):.3f}\", out_video,\n        )\n    run(\"ffmpeg\", \"-v\", \"error\", \"-y\", \"-i\", voice, \"-ar\", \"16000\", \"-ac\", \"1\", out_wav)\n\n\ndef _font(size):\n    from matplotlib import font_manager\n    from PIL import ImageFont\n\n    path = font_manager.findfont(font_manager.FontProperties(family=\"DejaVu Sans\", weight=\"bold\"))\n    return ImageFont.truetype(path, size)\n\n\ndef _text_card(text, path, size, fg, bg, top=None, bottom=None, W=1080, H=1920, max_w=900):\n    \"\"\"A full-frame transparent PNG with `text` wrapped in a rounded box (drawn with PIL, so ffmpeg needs no libass).\"\"\"\n    from PIL import Image, ImageDraw\n\n    font = _font(size)\n    img = Image.new(\"RGBA\", (W, H), (0, 0, 0, 0))\n    d = ImageDraw.Draw(img)\n    rows, row = [], \"\"\n    for word in text.split():\n        trial = f\"{row} {word}\".strip()\n        if d.textlength(trial, font=font) <= max_w - 64 or not row:\n            row = trial\n        else:\n            rows.append(row)\n            row = word\n    rows.append(row)\n    lh = int(size * 1.25)\n    bw = int(max(d.textlength(r, font=font) for r in rows)) + 64\n    bh = lh * len(rows) + 36\n    y0 = top if top is not None else H - bottom - bh\n    x0 = (W - bw) // 2\n    d.rounded_rectangle((x0, y0, x0 + bw, y0 + bh), radius=24, fill=bg)\n    for i, r in enumerate(rows):\n        d.text((W // 2, y0 + 18 + i * lh + lh // 2), r, font=font, fill=fg, anchor=\"mm\")\n    img.save(path)\n\n\ndef finish_ffmpeg(video, voice, lines, hook, out, ambient=None, ambient_volume=0.18):\n    \"\"\"Plain finishing without Remotion: 1080x1920, boxed subtitles, hook title, ambient bed.\"\"\"\n    dur = duration(voice)\n    with tempfile.TemporaryDirectory() as tmp:\n        cards = [(l[\"text\"], l[\"from\"], l[\"to\"], dict(size=60, fg=\"white\", bg=(12, 16, 10, 158), bottom=300)) for l in lines]\n        if hook:\n            cards.append((hook, 0.15, 4.0, dict(size=64, fg=(17, 17, 17), bg=\"white\", top=210)))\n        inputs, chain, last = [\"-i\", video, \"-i\", voice], [\"[0:v]scale=1080:1920:flags=lanczos,setsar=1[v0]\"], \"v0\"\n        for i, (text, a, b, style) in enumerate(cards):\n            png = Path(tmp) / f\"card{i}.png\"\n            _text_card(text, png, **style)\n            inputs += [\"-loop\", \"1\", \"-t\", f\"{dur:.3f}\", \"-i\", png]\n            chain.append(f\"[{last}][{i + 2}:v]overlay=0:0:enable='between(t,{a},{b})'[v{i + 1}]\")\n            last = f\"v{i + 1}\"\n        if ambient:\n            inputs += [\"-stream_loop\", \"-1\", \"-i\", ambient]\n            chain.append(f\"[{len(cards) + 2}:a]volume={ambient_volume}[bed];[1:a][bed]amix=inputs=2:duration=first:normalize=0[a]\")\n        else:\n            chain.append(\"[1:a]anull[a]\")\n        run(\n            \"ffmpeg\", \"-v\", \"error\", \"-y\", *inputs, \"-filter_complex\", \";\".join(chain),\n            \"-map\", f\"[{last}]\", \"-map\", \"[a]\", \"-c:v\", \"libx264\", \"-crf\", \"18\", \"-pix_fmt\", \"yuv420p\",\n            \"-c:a\", \"aac\", \"-b:a\", \"192k\", \"-t\", f\"{dur:.3f}\", out,\n        )\n\n\ndef loudnorm(src, out, target=-14):\n    \"\"\"Normalise to social-media loudness (about -14 LUFS), video copied as is.\"\"\"\n    run(\n        \"ffmpeg\", \"-v\", \"error\", \"-y\", \"-i\", src, \"-c:v\", \"copy\",\n        \"-af\", f\"loudnorm=I={target}:TP=-1.5:LRA=11\", \"-ar\", \"48000\", \"-c:a\", \"aac\", \"-b:a\", \"192k\", out,\n    )\n",
"studio/pipeline.py": "\"\"\"The studio's steps, shared by the web UI. Everything for one character lives in one folder on disk:\n\n  <home>/<character>/character.json, start.png, voice_ref.wav, voice_meta.json   (the \"character pack\")\n  <home>/<character>/videos/<id>/script.txt, hook.txt, voice.wav, lines.json, takes/, lipsync.mp4, final.mp4\n\nThe GPU-heavy steps (voice, video, lip-sync) run as separate processes in their own environments; long steps\nare generators that yield their log so the UI can show progress.\n\"\"\"\n\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport time\nfrom pathlib import Path\n\nfrom PIL import Image, ImageOps\n\nfrom media import duration, finish_ffmpeg, fit_to_audio, loudnorm\nfrom text import subtitle_lines\n\nHERE = Path(__file__).parent  # .../studio\nREMOTION = HERE.parent / \"remotion\"\nHOME = Path(os.environ.get(\"STUDIO_HOME\", \"/kaggle/working/studio\" if Path(\"/kaggle/working\").exists() else \"studio_home\"))\nWIDTH, HEIGHT = 480, 832\n\nTTS_ENV = Path(\"/tmp/venv_tts\")\nLATENTSYNC = Path(\"/tmp/LatentSync\")\nNODE = Path(\"/tmp/node-v20.18.0-linux-x64/bin\")\n\nTEST_LINE = (\n    \"Honestly, it wasn't anything clever. I just started doing the small things properly, \"\n    \"every day, and stopped waiting to feel motivated. That's it, really.\"\n)\n\nDEFAULT_NEGATIVE = (\n    \"talking, looking down, looking away, turning away, head turned, profile, walking, leaving frame, hands near \"\n    \"face, blurry eyes, closed eyes, glassy stare, face morphing, different person, deformed hands, extra fingers, \"\n    \"text, watermark, camera pan, zoom, cuts, plastic skin, airbrushed, blurry, low quality, static image, slow motion\"\n)\n\n\n# ------------------------------------------------------------------ helpers\n\ndef slug(name):\n    return re.sub(r\"[^a-z0-9]+\", \"-\", name.lower()).strip(\"-\") or \"character\"\n\n\ndef stream(cmd, cwd=None, env=None):\n    \"\"\"Run a shell command, yielding its output line by line; raises if it fails.\"\"\"\n    # unbuffered, so a child's prints reach the UI as they happen instead of in one burst at the end\n    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, \"PYTHONUNBUFFERED\": \"1\", **(env or {})},\n                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n    tail = []\n    for line in p.stdout:\n        line = line.rstrip()\n        if line:\n            tail = (tail + [line])[-15:]\n            yield line\n    if p.wait():\n        raise RuntimeError(f\"step failed (exit {p.returncode}). Last output:\\n\" + \"\\n\".join(tail))\n\n\nclass Log:\n    \"\"\"Collects log lines; each step yields `log.text` so the UI shows the whole history.\"\"\"\n\n    def __init__(self):\n        self.lines = []\n\n    def add(self, line):\n        if \"%|\" in line and self.lines and \"%|\" in self.lines[-1]:\n            self.lines[-1] = line  # a progress bar redrawing itself: keep only its latest state\n        else:\n            self.lines.append(line)\n        self.lines = self.lines[-400:]\n        return self.text\n\n    @property\n    def text(self):\n        return \"\\n\".join(self.lines)\n\n\n# ------------------------------------------------------------------ character\n\ndef char_dir(name):\n    d = HOME / slug(name)\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\ndef list_characters():\n    return sorted(p.name for p in HOME.glob(\"*\") if (p / \"character.json\").exists()) if HOME.exists() else []\n\n\ndef save_character(fields, start_image=None):\n    d = char_dir(fields[\"name\"])\n    (d / \"character.json\").write_text(json.dumps(fields, indent=2))\n    if start_image:\n        img = ImageOps.exif_transpose(Image.open(start_image)).convert(\"RGB\")\n        w, h = img.size  # centre-crop to 480:832, then resize\n        if w / h > WIDTH / HEIGHT:\n            nw = int(h * WIDTH / HEIGHT)\n            img = img.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))\n        else:\n            nh = int(w * HEIGHT / WIDTH)\n            img = img.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))\n        img.resize((WIDTH, HEIGHT), Image.LANCZOS).save(d / \"start.png\")\n    return d\n\n\ndef load_character(char_slug):\n    d = HOME / char_slug\n    fields = json.loads((d / \"character.json\").read_text())\n    start = d / \"start.png\"\n    return fields, str(start) if start.exists() else None\n\n\ndef import_pack(zip_path):\n    \"\"\"Unpack a character_pack.zip (from export_pack) into the studio; returns the character's slug.\"\"\"\n    tmp = HOME / \"_import\"\n    shutil.rmtree(tmp, ignore_errors=True)\n    shutil.unpack_archive(zip_path, tmp)\n    found = next(tmp.rglob(\"character.json\"))\n    name = json.loads(found.read_text())[\"name\"]\n    dest = char_dir(name)\n    for f in found.parent.iterdir():\n        if f.is_file():\n            shutil.copy(f, dest)\n    shutil.rmtree(tmp)\n    return dest.name\n\n\ndef export_pack(name):\n    d = char_dir(name)\n    files = [f for f in (\"character.json\", \"start.png\", \"voice_ref.wav\", \"voice_meta.json\") if (d / f).exists()]\n    stage = HOME / \"_export\" / d.name\n    shutil.rmtree(HOME / \"_export\", ignore_errors=True)\n    stage.mkdir(parents=True)\n    for f in files:\n        shutil.copy(d / f, stage)\n    return shutil.make_archive(str(d / f\"{d.name}_character_pack\"), \"zip\", HOME / \"_export\", d.name)\n\n\ndef video_prompt(fields):\n    return (\n        \"Real vertical smartphone footage, waist-up, phone propped on a shelf, still camera. \"\n        f\"{fields.get('look', '').strip().capitalize()}, in {fields.get('setting', '').strip()}. Faces the camera the whole time \"\n        \"and looks into the lens as if about to speak to a friend: small natural head movements, a slight nod, gentle \"\n        \"weight shifts, relaxed shoulders, natural blinks, warm engaged eyes, mouth relaxed and closed. \"\n        f\"{fields.get('light', '').strip().capitalize()}. Sharp, natural colour, real skin texture.\"\n    )\n\n\n# ------------------------------------------------------------------ ideas and script (local LLM, in this process)\n\n_writer = None\n\n\ndef _get_writer(fields):\n    global _writer\n    from ideas import Writer\n\n    if _writer is None:\n        _writer = Writer(fields)\n    _writer.character = fields\n    return _writer\n\n\ndef release_writer():\n    \"\"\"Free the LLM's GPU memory before a GPU-heavy step.\"\"\"\n    global _writer\n    if _writer is not None:\n        _writer.close()\n        _writer = None\n\n\ndef ideas(fields, theme, n=8):\n    return _get_writer(fields).ideas(n, theme)\n\n\ndef write_script(fields, idea, seconds):\n    return _get_writer(fields).script(idea, seconds)\n\n\ndef new_video(name, script, hook):\n    d = char_dir(name) / \"videos\" / time.strftime(\"%Y%m%d-%H%M%S\")\n    d.mkdir(parents=True)\n    (d / \"script.txt\").write_text(script.strip())\n    (d / \"hook.txt\").write_text(hook.strip())\n    return str(d)\n\n\n# ------------------------------------------------------------------ voice\n\ndef _ensure_tts(log):\n    if not shutil.which(\"sox\"):  # the voice library looks for SoX at start-up; install it to keep the log clean\n        subprocess.run(\"apt-get -qq install -y sox\", shell=True, capture_output=True)\n    # The .ready marker is written only after a complete, verified install, so a half-finished install\n    # (interrupted, out of disk, network blip) is redone instead of silently reused.\n    if (TTS_ENV / \".ready\").exists():\n        return\n    yield log.add(\"Installing the voice model environment (first time only, ~5 min)...\")\n    shutil.rmtree(TTS_ENV, ignore_errors=True)\n    # torch from the CUDA 12.8 index: the newest PyPI build targets CUDA 13, which Kaggle's driver may not run.\n    for line in stream(\n        f\"uv venv -q -p 3.10 {TTS_ENV} && \"\n        f\"uv pip install -p {TTS_ENV} --index-url https://download.pytorch.org/whl/cu128 torch torchaudio && \"\n        f\"uv pip install -p {TTS_ENV} qwen-tts soundfile numpy && \"\n        f\"{TTS_ENV}/bin/python -c 'import torch, qwen_tts, soundfile, numpy; \"\n        \"print(\\\"voice env OK, torch\\\", torch.__version__, \\\"cuda\\\", torch.cuda.is_available())'\"\n    ):\n        if not line.startswith((\" +\", \" -\", \"Resolved\", \"Prepared\", \"Installed\", \"Downloading\", \"Uninstalled\")):\n            yield log.add(line)\n    (TTS_ENV / \".ready\").touch()\n\n\ndef _tts(args):\n    return f\"{TTS_ENV}/bin/python {HERE}/tts.py {args}\"\n\n\ndef saved_voice(name):\n    d = char_dir(name)\n    meta = d / \"voice_meta.json\"\n    if (d / \"voice_ref.wav\").exists() and meta.exists():\n        return str(d / \"voice_ref.wav\"), json.loads(meta.read_text())\n    return None, None\n\n\ndef design_voice(name, description, language, takes, dtype=\"float32\"):\n    \"\"\"Generator: yields (log, list of take paths).\"\"\"\n    release_writer()\n    log = Log()\n    for t in _ensure_tts(log):\n        yield t, []\n    out = char_dir(name) / \"voice_takes\"\n    shutil.rmtree(out, ignore_errors=True)\n    spec = char_dir(name) / \"voice_spec.txt\"\n    spec.write_text(description.strip())\n    yield log.add(f\"Designing {takes} voice takes...\"), []\n    line = TEST_LINE.replace('\"', \"'\")\n    for l in stream(_tts(f'--lang {language} --dtype {dtype} design --spec \"{spec}\" --line \"{line}\" --takes {takes} --out \"{out}\"')):\n        # hand over each take as soon as it exists, so it can be played while the rest are made\n        yield log.add(l), [str(out / f\"take_{i}.wav\") for i in range(1, takes + 1) if (out / f\"take_{i}.wav\").exists()]\n    yield log.add(\"Listen and pick the one that sounds like your character.\"), [str(out / f\"take_{i}.wav\") for i in range(1, takes + 1)]\n\n\ndef lock_voice(name, take_path, description, language):\n    d = char_dir(name)\n    shutil.copy(take_path, d / \"voice_ref.wav\")\n    (d / \"voice_meta.json\").write_text(\n        json.dumps({\"description\": description.strip(), \"ref_text\": TEST_LINE, \"language\": language}, indent=2)\n    )\n    return str(d / \"voice_ref.wav\")\n\n\ndef speak(name, video_dir, dtype=\"float32\"):\n    \"\"\"Generator: yields (log, voice path or None).\"\"\"\n    release_writer()\n    log = Log()\n    ref, meta = saved_voice(name)\n    if not ref:\n        raise RuntimeError(\"No voice saved for this character yet: design one and click 'Use this voice'.\")\n    for t in _ensure_tts(log):\n        yield t, None\n    vd = Path(video_dir)\n    (vd / \"ref_text.txt\").write_text(meta[\"ref_text\"])\n    yield log.add(\"Speaking the script in the character's voice...\"), None\n    cmd = _tts(f'--lang {meta[\"language\"]} --dtype {dtype} speak --script \"{vd}/script.txt\" --ref \"{ref}\" '\n               f'--ref-text \"{vd}/ref_text.txt\" --out \"{vd}\"')\n    for l in stream(cmd):\n        yield log.add(l), None\n    lines = subtitle_lines(json.loads((vd / \"sentences.json\").read_text()))\n    (vd / \"lines.json\").write_text(json.dumps(lines, indent=2))\n    yield log.add(f\"Voice ready: {duration(vd / 'voice.wav'):.1f}s, {len(lines)} subtitle lines.\"), str(vd / \"voice.wav\")\n\n\n# ------------------------------------------------------------------ video takes\n\ndef make_takes(name, video_dir, prompt, negative, seeds, steps):\n    \"\"\"Generator: yields (log, list of take paths).\"\"\"\n    release_writer()\n    log = Log()\n    vd = Path(video_dir)\n    (vd / \"prompt.txt\").write_text(prompt.strip())\n    (vd / \"negative.txt\").write_text(negative.strip())\n    takes = vd / \"takes\"\n    yield log.add(f\"Making {len(seeds)} takes (about 20 min each on a T4)...\"), []\n    cmd = (f'python {HERE}/wan.py --image \"{char_dir(name)}/start.png\" --prompt \"{vd}/prompt.txt\" '\n           f'--negative \"{vd}/negative.txt\" --seeds {\" \".join(map(str, seeds))} --steps {steps} --out \"{takes}\"')\n    for l in stream(cmd):\n        yield log.add(l), sorted(map(str, takes.glob(\"take_seed*.mp4\")))\n    yield log.add(\"Takes ready. Pick the ones to use.\"), sorted(map(str, takes.glob(\"take_seed*.mp4\")))\n\n\n# ------------------------------------------------------------------ lip-sync\n\ndef _ensure_latentsync(log):\n    if not (LATENTSYNC / \".ready\").exists():\n        yield log.add(\"Installing LatentSync (first time only, ~5 min)...\")\n        for line in stream(\n            f\"rm -rf {LATENTSYNC} && git clone -q https://github.com/bytedance/LatentSync.git {LATENTSYNC} && \"\n            f\"(apt-get -qq install -y libgl1 > /dev/null 2>&1 || true) && cd {LATENTSYNC} && \"\n            \"uv venv -q -p 3.10 .venv && uv pip install -q -p .venv -r requirements.txt && \"\n            \".venv/bin/huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt latentsync_unet.pt \"\n            \"--local-dir checkpoints\"\n        ):\n            yield log.add(line)\n        (LATENTSYNC / \".ready\").touch()\n\n\ndef lipsync(video_dir, take_paths, keep_seconds, fp16=False, steps=20):\n    \"\"\"Generator: yields (log, lip-synced video path or None).\"\"\"\n    release_writer()\n    log = Log()\n    vd = Path(video_dir)\n    if not take_paths:\n        raise RuntimeError(\"Pick at least one take.\")\n    yield log.add(\"Chaining the takes to the length of the voice...\"), None\n    keep = {p: keep_seconds for p in take_paths} if keep_seconds else None\n    fit_to_audio(vd / \"voice.wav\", take_paths, vd / \"lipsync_input.mp4\", vd / \"lipsync_voice.wav\", keep=keep)\n    for t in _ensure_latentsync(log):\n        yield t, None\n    sed = \"s/get_device_capability()\\\\[0\\\\] > 7/get_device_capability()[0] >= 7/\" if fp16 else \\\n          \"s/get_device_capability()\\\\[0\\\\] >= 7/get_device_capability()[0] > 7/\"\n    subprocess.run([\"sed\", \"-i\", sed, str(LATENTSYNC / \"scripts/inference.py\")], check=True)\n    yield log.add(f\"Lip-syncing {duration(vd / 'lipsync_voice.wav'):.0f}s (about 1-3 min per second)...\"), None\n    out = vd / \"lipsync.mp4\"\n    cmd = (\".venv/bin/python -m scripts.inference --unet_config_path configs/unet/stage2.yaml \"\n           f\"--inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps {steps} --guidance_scale 1.5 \"\n           f'--seed 1247 --enable_deepcache --video_path \"{vd}/lipsync_input.mp4\" --audio_path \"{vd}/lipsync_voice.wav\" '\n           f'--video_out_path \"{out}\" --temp_dir /tmp/ls_temp')\n    for l in stream(cmd, cwd=LATENTSYNC, env={\"MPLBACKEND\": \"Agg\"}):\n        yield log.add(l), None\n    for f in (\"lipsync_input.mp4\", \"lipsync_voice.wav\"):\n        (vd / f).unlink(missing_ok=True)\n    yield log.add(\"Lip-sync done.\"), str(out)\n\n\n# ------------------------------------------------------------------ finishing\n\ndef _ensure_node(log):\n    \"\"\"Use the system's Node.js if there is one; otherwise (Kaggle) download a Linux build to /tmp.\"\"\"\n    if shutil.which(\"node\") or (NODE / \"node\").exists():\n        return\n    yield log.add(\"Installing Node.js (first time only)...\")\n    for line in stream(f\"curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz | tar -xJ -C {NODE.parent.parent}\"):\n        yield log.add(line)\n\n\ndef finish(video_dir, hook, ambient_volume, engine=\"remotion\"):\n    \"\"\"Generator: yields (log, final video path or None).\"\"\"\n    log = Log()\n    vd = Path(video_dir)\n    voice, lipsynced = vd / \"voice.wav\", vd / \"lipsync.mp4\"\n    lines = json.loads((vd / \"lines.json\").read_text())\n    raw = vd / \"finished_raw.mp4\"\n    ambient = REMOTION / \"public/audio/ambient.wav\"\n    env = {\"PATH\": f\"{NODE}:{os.environ['PATH']}\"}\n    try:  # Node.js renders with Remotion and generates the ambient bed\n        for t in _ensure_node(log):\n            yield t, None\n        if not ambient.exists():\n            for l in stream(\"node tools/make-audio.mjs\", cwd=REMOTION, env=env):\n                yield log.add(l), None\n    except RuntimeError as e:\n        yield log.add(f\"Node.js unavailable ({e}); finishing with ffmpeg, without ambient sound.\"), None\n        engine = \"ffmpeg\"\n\n    if engine == \"remotion\":\n        try:\n            yield log.add(\"Installing headless-browser libraries...\"), None\n            for lib in (\"libnss3 libdbus-1-3 libatk1.0-0 libgbm1 libasound2 libxrandr2 libxkbcommon0 libxfixes3 \"\n                        \"libxcomposite1 libxdamage1 libatk-bridge2.0-0 libpango-1.0-0 libcairo2 libcups2\").split():\n                subprocess.run(f\"apt-get -qq install -y {lib}\", shell=True, capture_output=True)\n            (REMOTION / \"public/video\").mkdir(parents=True, exist_ok=True)\n            shutil.copy(lipsynced, REMOTION / \"public/video/talking.mp4\")\n            shutil.copy(voice, REMOTION / \"public/audio/voice.wav\")\n            (REMOTION / \"props.json\").write_text(json.dumps({\n                \"video\": \"video/talking.mp4\", \"voice\": \"audio/voice.wav\", \"lines\": lines, \"hook\": hook,\n                \"durationSeconds\": duration(voice), \"ambientVolume\": ambient_volume,\n            }))\n            yield log.add(\"Rendering with Remotion...\"), None\n            for l in stream(\"npm install --silent --no-audit --no-fund\", cwd=REMOTION, env=env):\n                yield log.add(l), None\n            for l in stream(f'npx remotion render src/index.ts Talking \"{raw}\" --props=props.json --codec=h264 '\n                            \"--crf=18 --log=error\", cwd=REMOTION, env=env):\n                yield log.add(l), None\n        except RuntimeError as e:\n            yield log.add(f\"Remotion failed ({e}); finishing with ffmpeg instead.\"), None\n            engine = \"ffmpeg\"\n    if engine == \"ffmpeg\":\n        yield log.add(\"Finishing with ffmpeg...\"), None\n        finish_ffmpeg(lipsynced, voice, lines, hook, raw, ambient=ambient if ambient_volume and ambient.exists() else None,\n                      ambient_volume=ambient_volume)\n    final = vd / \"final.mp4\"\n    loudnorm(raw, final)\n    raw.unlink()\n    yield log.add(f\"Done: {final}\"), str(final)\n",
"studio/text.py": "\"\"\"Script -> sentences (with pauses) for the voice, and sentence timings -> short subtitle lines.\"\"\"\n\nimport re\n\nPAUSE_SENTENCE = 0.35  # seconds of silence between sentences\nPAUSE_PARAGRAPH = 0.7  # between paragraphs (blank line in the script)\nLEAD_IN = 0.4  # silence before the first word\nMAX_WORDS = 5  # words per subtitle line\n\n\ndef split_script(text):\n    \"\"\"-> list of (sentence, pause_after) in reading order.\"\"\"\n    out = []\n    paragraphs = [p.strip() for p in re.split(r\"\\n\\s*\\n\", text) if p.strip()]\n    for pi, para in enumerate(paragraphs):\n        sentences = [s.strip() for s in re.split(r\"(?<=[.!?])\\s+\", \" \".join(para.split())) if s.strip()]\n        for si, s in enumerate(sentences):\n            last_in_para = si == len(sentences) - 1\n            last_overall = last_in_para and pi == len(paragraphs) - 1\n            out.append((s, 0.0 if last_overall else PAUSE_PARAGRAPH if last_in_para else PAUSE_SENTENCE))\n    return out\n\n\ndef _chunk(words):\n    n = -(-len(words) // MAX_WORDS)\n    size = -(-len(words) // n)\n    return [\" \".join(words[i : i + size]) for i in range(0, len(words), size)]\n\n\ndef subtitle_lines(sentences):\n    \"\"\"[{text, from, to}] per sentence -> shorter lines, split at commas first, timed by share of characters.\"\"\"\n    lines = []\n    for s in sentences:\n        pieces = [p for clause in re.split(r\"(?<=,) \", s[\"text\"]) for p in _chunk(clause.split(\" \"))]\n        total = sum(len(p) for p in pieces)\n        t = s[\"from\"]\n        for p in pieces:\n            d = (s[\"to\"] - s[\"from\"]) * len(p) / total\n            lines.append({\"text\": p, \"from\": round(t, 2), \"to\": round(t + d, 2)})\n            t += d\n    return lines\n",
"studio/tts.py": "\"\"\"Character voice with Qwen3-TTS on a CUDA GPU. Runs in its own environment (see the notebook).\n\n  design: a few takes of one neutral line, in a voice built from a written description; pick one.\n  speak:  clone the picked take for every sentence of a script, so the whole voice-over is one voice.\n          Writes voice.wav and sentences.json ([{text, from, to}] in seconds).\n\n  python tts.py design --spec spec.txt --line \"...\" --takes 4 --out takes/\n  python tts.py speak --script script.txt --ref voice_ref.wav --ref-text ref_text.txt --out out/\n\"\"\"\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom qwen_tts import Qwen3TTSModel\n\nsys.path.insert(0, str(Path(__file__).parent))\nfrom text import LEAD_IN, split_script  # noqa: E402\n\nDESIGN_MODEL = \"Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign\"\nCLONE_MODEL = \"Qwen/Qwen3-TTS-12Hz-1.7B-Base\"\nDTYPES = {\"float32\": torch.float32, \"float16\": torch.float16, \"bfloat16\": torch.bfloat16}\n\n\ndef load(name, dtype):\n    # T4/P100 have no fast bf16 and no FlashAttention 2, so default to plain attention.\n    return Qwen3TTSModel.from_pretrained(name, device_map=\"cuda:0\", dtype=DTYPES[dtype], attn_implementation=\"sdpa\")\n\n\ndef design(args):\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    model = load(DESIGN_MODEL, args.dtype)\n    spec = Path(args.spec).read_text().strip()\n    for i in range(1, args.takes + 1):\n        print(f\"Making take {i} of {args.takes}...\", flush=True)\n        torch.manual_seed(100 + i)\n        wavs, sr = model.generate_voice_design(text=args.line, language=args.lang, instruct=spec)\n        sf.write(out / f\"take_{i}.wav\", np.asarray(wavs[0], dtype=np.float32), sr)\n        print(f\"Take {i} of {args.takes} ready.\", flush=True)\n\n\ndef speak(args):\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    model = load(CLONE_MODEL, args.dtype)\n    prompt = model.create_voice_clone_prompt(ref_audio=args.ref, ref_text=Path(args.ref_text).read_text().strip())\n\n    chunks, sentences, sr = [], [], None\n    t = LEAD_IN\n    for text, pause in split_script(Path(args.script).read_text()):\n        print(f\"Speaking: {text}\", flush=True)\n        torch.manual_seed(7)\n        wavs, sr = model.generate_voice_clone(text=text, language=args.lang, voice_clone_prompt=prompt)\n        audio = np.asarray(wavs[0], dtype=np.float32)\n        if not chunks:\n            chunks.append(np.zeros(int(LEAD_IN * sr), dtype=np.float32))\n        dur = len(audio) / sr\n        sentences.append({\"text\": text, \"from\": round(t, 2), \"to\": round(t + dur, 2)})\n        chunks += [audio, np.zeros(int(pause * sr), dtype=np.float32)]\n        t += dur + pause\n        print(f\"  {dur:4.1f}s  {text}\")\n\n    chunks.append(np.zeros(int(0.5 * sr), dtype=np.float32))  # tail room\n    audio = np.concatenate(chunks)\n    audio = audio / max(1e-6, np.abs(audio).max()) * 0.89\n    sf.write(out / \"voice.wav\", audio, sr)\n    (out / \"sentences.json\").write_text(json.dumps(sentences, indent=2))\n    print(f\"voice: {len(audio) / sr:.1f}s, {len(sentences)} sentences\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--lang\", default=\"English\")\n    ap.add_argument(\"--dtype\", default=\"float32\", choices=DTYPES)\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n    d = sub.add_parser(\"design\")\n    d.add_argument(\"--spec\", required=True)\n    d.add_argument(\"--line\", required=True)\n    d.add_argument(\"--takes\", type=int, default=4)\n    d.add_argument(\"--out\", required=True)\n    s = sub.add_parser(\"speak\")\n    s.add_argument(\"--script\", required=True)\n    s.add_argument(\"--ref\", required=True)\n    s.add_argument(\"--ref-text\", required=True)\n    s.add_argument(\"--out\", required=True)\n    args = ap.parse_args()\n    design(args) if args.cmd == \"design\" else speak(args)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"studio/wan.py": "\"\"\"Base clips with Wan 2.2 TI2V-5B (GGUF Q8) on a 16 GB GPU: one ~5 s take per seed, from one start frame.\n\n  python wan.py --image start.png --prompt prompt.txt --negative negative.txt --seeds 42 7 --out takes/\n\nA take that already has its mp4 is skipped, so re-running resumes. Latents are saved before decoding, so a\ncrash in the decode loses no generation time.\n\"\"\"\n\nimport argparse\nimport gc\nimport os\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom diffusers import AutoencoderKLWan, GGUFQuantizationConfig, WanImageToVideoPipeline, WanTransformer3DModel\nfrom diffusers.utils import export_to_video\nfrom huggingface_hub import HfApi, hf_hub_download\nfrom PIL import Image\nfrom transformers import AutoTokenizer, BitsAndBytesConfig, UMT5EncoderModel\n\nBASE = \"Wan-AI/Wan2.2-TI2V-5B-Diffusers\"\nDTYPE = torch.float16  # T4/P100 have no fast bf16\nWIDTH, HEIGHT = 480, 832\nNUM_FRAMES, FPS = 121, 24\n\n\ndef encode_prompt(prompt, negative):\n    # 8-bit text encoder, freed straight after so the video model has the GPU to itself\n    tokenizer = AutoTokenizer.from_pretrained(BASE, subfolder=\"tokenizer\")\n    text_encoder = UMT5EncoderModel.from_pretrained(\n        BASE, subfolder=\"text_encoder\", quantization_config=BitsAndBytesConfig(load_in_8bit=True),\n        torch_dtype=DTYPE, device_map={\"\": 0}, low_cpu_mem_usage=True,\n    )\n    te = WanImageToVideoPipeline.from_pretrained(\n        BASE, tokenizer=tokenizer, text_encoder=text_encoder, transformer=None, vae=None,\n        image_encoder=None, image_processor=None, torch_dtype=DTYPE,\n    )\n    with torch.no_grad():\n        pe, ne = te.encode_prompt(\n            prompt=prompt, negative_prompt=negative, do_classifier_free_guidance=True,\n            num_videos_per_prompt=1, max_sequence_length=512, device=\"cuda\",\n        )\n    del te, text_encoder\n    gc.collect()\n    torch.cuda.empty_cache()\n    return pe.to(\"cpu\", DTYPE), ne.to(\"cpu\", DTYPE)\n\n\ndef load_pipe():\n    api = HfApi()\n    repo = file = None\n    for m in api.list_models(search=\"Wan2.2-TI2V-5B-GGUF\", sort=\"downloads\", limit=10):\n        pick = [f for f in api.list_repo_files(m.id) if f.endswith(\".gguf\") and \"Q8_0\" in f]\n        if pick:\n            repo, file = m.id, pick[0]\n            break\n    assert repo, \"No Wan2.2-TI2V-5B GGUF found on the Hub\"\n    print(\"video model:\", repo, file, flush=True)\n    transformer = WanTransformer3DModel.from_single_file(\n        hf_hub_download(repo, file), quantization_config=GGUFQuantizationConfig(compute_dtype=DTYPE),\n        config=BASE, subfolder=\"transformer\", torch_dtype=DTYPE,\n    )\n    vae = AutoencoderKLWan.from_pretrained(BASE, subfolder=\"vae\", torch_dtype=torch.float32)\n    pipe = WanImageToVideoPipeline.from_pretrained(\n        BASE, transformer=transformer, vae=vae, text_encoder=None, tokenizer=None,\n        image_encoder=None, image_processor=None, torch_dtype=DTYPE,\n    )\n    pipe.enable_model_cpu_offload()\n    pipe.vae.enable_tiling()\n    return pipe\n\n\ndef decode(pipe, latents):\n    v = pipe.vae.to(\"cuda\", torch.float16)\n    lat = latents.to(\"cuda\", torch.float16)\n    mean = torch.tensor(v.config.latents_mean).view(1, v.config.z_dim, 1, 1, 1).to(lat)\n    inv_std = 1.0 / torch.tensor(v.config.latents_std).view(1, v.config.z_dim, 1, 1, 1).to(lat)\n    with torch.no_grad():\n        video = v.decode(lat / inv_std + mean, return_dict=False)[0]\n    v.to(\"cpu\")\n    torch.cuda.empty_cache()\n    return pipe.video_processor.postprocess_video(video.float(), output_type=\"np\")[0]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--image\", required=True)\n    ap.add_argument(\"--prompt\", required=True)\n    ap.add_argument(\"--negative\", required=True)\n    ap.add_argument(\"--seeds\", type=int, nargs=\"+\", required=True)\n    ap.add_argument(\"--steps\", type=int, default=40)\n    ap.add_argument(\"--guidance\", type=float, default=5.0)\n    ap.add_argument(\"--out\", required=True)\n    args = ap.parse_args()\n\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    todo = [s for s in args.seeds if not (out / f\"take_seed{s}.mp4\").exists()]\n    if not todo:\n        print(\"all takes already made\")\n        return\n    img = Image.open(args.image).convert(\"RGB\").resize((WIDTH, HEIGHT), Image.LANCZOS)\n    pe, ne = encode_prompt(Path(args.prompt).read_text(), Path(args.negative).read_text())\n    pipe = load_pipe()\n\n    for seed in todo:\n        t0 = time.time()\n        lat_path = out / f\"latents_seed{seed}.pt\"\n        print(f\"take seed {seed}: generating ({args.steps} steps)...\", flush=True)\n        if not lat_path.exists():\n            with torch.no_grad():\n                lat = pipe(\n                    image=img, prompt_embeds=pe.to(\"cuda\"), negative_prompt_embeds=ne.to(\"cuda\"),\n                    height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES, num_inference_steps=args.steps,\n                    guidance_scale=args.guidance, generator=torch.Generator(\"cuda\").manual_seed(seed),\n                    output_type=\"latent\",\n                ).frames\n            torch.save(lat.cpu(), lat_path)\n        export_to_video(decode(pipe, torch.load(lat_path)), str(out / f\"take_seed{seed}.mp4\"), fps=FPS)\n        os.remove(lat_path)\n        print(f\"take seed {seed}: done in {(time.time() - t0) / 60:.1f} min\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n"
}

import os, sys
from pathlib import Path
for rel, text in FILES.items():
    p = Path("/tmp/studio") / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
sys.path.insert(0, "/tmp/studio/studio")
print("unpacked", len(FILES), "files")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Keep Kaggle's own torch: pin it so no package below can swap it out of the running kernel
# (that breaks it with "module 'torch' has no attribute '_utils'").
!python -c "import importlib.metadata as m; [print(f'{p}=={m.version(p)}') for p in ('torch','torchvision','torchaudio') if any(d.metadata['Name'].lower()==p for d in m.distributions())]" > /tmp/torch-pin.txt
!pip -q install -U -c /tmp/torch-pin.txt "gradio>=6,<7" "diffusers>=0.35" "transformers>=4.51" accelerate gguf bitsandbytes ftfy imageio imageio-ffmpeg sentencepiece uv
!python -c "import torch; print('torch OK', torch.__version__, torch.cuda.is_available())"

If a cell fails with `torch has no attribute '_utils'`: **Run → Restart & clear cell outputs**, then run all again.

## Launch the studio

In [ ]:
PASSWORD = "change-me"   # the public link is open to anyone who has it, so protect your GPU with a password

assert PASSWORD != "change-me", "Set your own PASSWORD first."
from app import launch
demo = launch(share=True, password=PASSWORD)